In [2]:
import polars as pl
import plotly.express as px

In [ ]:
data_folder_path = "G:/coding/python/web_scraping/Youm7/youm7_scrape/data/intermediate"
file_name = "urgent.parquet"
file_path = f"{data_folder_path}/{file_name}"

In [3]:
lf = pl.scan_parquet(r"G:\coding\python\web_scraping\Youm7\youm7_scrape\data\intermediate\your_horoscope_today.parquet")

In [3]:
lf.columns

C:\Users\hp\AppData\Local\Temp\ipykernel_19532\3701027225.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  lf.columns


['article_id',
 'url',
 'category',
 'title',
 'publish_date',
 'author',
 'content',
 'tags',
 'media',
 'date_parsed']

In [4]:
for df in lf.count().collect_batches(maintain_order=False):
    print(df)

Exception ignored in: <function LazyFrame.collect_batches.<locals>.BatchCollector.__del__ at 0x000001E579A9E8C0>
Traceback (most recent call last):
  File "C:\Users\hp\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\LocalCache\local-packages\Python310\site-packages\polars\lazyframe\frame.py", line 3882, in __del__
    self._fut.result()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\concurrent\futures\_base.py", line 451, in result
    return self.__get_result()
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\concurrent\futures\_base.py", line 403, in __get_result
    raise self._exception
  File "C:\Program Files\WindowsApps\PythonSoftwareFoundation.Python.3.10_3.10.3056.0_x64__qbz5n2kfra8p0\lib\concurrent\futures\thread.py", line 58, in run
    result = self.fn(*self.args, **self.kwargs)
  File "C:\Users\hp\AppData\Local\Packages\PythonS

In [28]:
lf.select("date_parsed", "publish_date").collect()

date_parsed,publish_date
datetime[μs],str
2026-03-18 18:10:00,"""الأربعاء، 18 مارس 2026 06:10 م"""
2026-03-18 17:31:00,"""الأربعاء، 18 مارس 2026 05:31 م"""
2026-03-18 17:52:00,"""الأربعاء، 18 مارس 2026 05:52 م"""
2026-03-18 16:55:00,"""الأربعاء، 18 مارس 2026 04:55 م"""
2026-03-18 17:38:00,"""الأربعاء، 18 مارس 2026 05:38 م"""
…,…
2015-12-31 23:15:00,"""الخميس، 31 ديسمبر 2015 11:15 م"""
2019-03-13 11:08:00,"""الأربعاء، 13 مارس 2019 11:08 ص"""
2019-02-21 08:00:00,"""الخميس، 21 فبراير 2019 08:00 ص"""


In [3]:
lf.columns

C:\Users\hp\AppData\Local\Temp\ipykernel_16696\3701027225.py:1: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  lf.columns


['article_id',
 'url',
 'category',
 'title',
 'publish_date',
 'author',
 'content',
 'tags',
 'media',
 'date_parsed']

In [11]:
# Use explode() to turn the list of tags into individual rows
top_tags = (
    lf.explode("tags")
    .filter(pl.col("tags").is_not_null())
    .group_by("tags")
    .agg(pl.len().alias("frequency"))
    .sort("frequency", descending=True)
    .head(20) # Get top 20
    .collect()
)
fig = px.pie(top_tags, values="frequency", names="tags", title="Top 20 Trending Tags")
fig.show()

In [5]:
top_tags.head()

tags,frequency
str,u32
"""اخبار الفن""",7320
"""مسلسلات رمضان""",3353
"""مسلسلات رمضان 2023""",2745
"""Watch iT""",2508
"""مسلسلات المتحدة""",2461


In [ ]:
word = ""
top_tags.filter(pl.col("tags") == word)

tags,frequency
str,u32
"""السيسي""",4957


In [12]:
# 1. Extract Hour and Weekday
heatmap_data = (
    lf.with_columns([
        pl.col("date_parsed").dt.hour().alias("hour"),
        pl.col("date_parsed").dt.weekday().alias("weekday") # 1=Mon, 5=Fri
    ])
    .group_by(["weekday", "hour"])
    .agg(pl.len().alias("article_count"))
    .collect()
)

# 2. Plotting
fig = px.density_heatmap(
    heatmap_data, 
    x="hour", 
    y="weekday", 
    z="article_count",
    labels={'weekday': 'Day of Week (1=Mon, 7=Sun)', 'hour': 'Hour of Day'},
    title="Youm7 Publishing Intensity Heatmap"
)
fig.show()

In [6]:
lf.count().collect()

article_id,url,category,title,publish_date,author,content,tags,media,date_parsed
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
48187,48187,48187,48187,48187,4370,48187,48187,48187,48187


In [ ]:
lf.select("content").sql("select * from self where content like \'%الحمل%\'").collect()

content
str
"""يتسم مولود برج الحمل بالعديد م…"
"""يتميز مولود برج الحمل بروح الم…"
"""برج الحمل الناري من الأبراج ال…"
"""نقدم حظك اليوم وتوقعات الأبراج…"
"""يبدأ برج الحمل من يوم 21 مارس …"
…
"""بعض الأشخاص يرفضون الغموض والم…"
"""مع أجواء البهجة التي ترافق أيا…"
"""تتغير الحظوظ من فترة لأخرى، وي…"
